## Transformer机器翻译

In [ ]:
import jieba
from win32inetcon import SECURITY_INTERNET_MASK

data = [
    ("你好，今天天气真好！", "Hello, the weather is nice today!"),
    ("你吃饭了吗？", "Have you eaten yet?"),
    ("深度学习很有趣。", "Deep learning is interesting."),
    ("我们一起学习吧。", "We are learning together."),
    ("这是一个测试案例。", "This is a test example.")
]


def tokenize_chinese(text):
    return list(jieba.cut(text))


def tokenize_english(text):
    return text.lower().split()


chinese_vocab = [tokenize_chinese(pair[0]) for pair in data]
english_vocab = [tokenize_english(pair[1]) for pair in data]

chinese_sentences = [tokenize_chinese(pair[0]) for pair in data]
english_sentences = [tokenize_english(pair[1]) for pair in data]


In [ ]:
from collections import Counter

special_tokens = ['<PAD>', '<UNK>', '<BOS>', '<EOS>']


def build_vocab(sentences):
    counter = Counter()

    for sentence in sentences:
        for word in sentence:
            counter[word] += 1

    vocab = special_tokens.copy()
    for word, count in counter.items():
        if word not in special_tokens:
            vocab.append(word)

    word_to_idx = {word: idx for idx, word in enumerate(vocab)}
    return word_to_idx, vocab


chinese_word_to_idx, chinese_vocab = build_vocab([sentence for sentence in chinese_sentences])
english_word_to_idx, english_vocab = build_vocab([sentence for sentence in english_sentences])

print(chinese_vocab)
print(english_vocab)
print(chinese_word_to_idx)
print(english_word_to_idx)

In [ ]:
import torch

ch_vocab_size = len(chinese_vocab)
en_vocab_size = len(english_vocab)
hidden_size = 256
batch_size = 2
Learning_rate = 0.0001
vocab_size = max(ch_vocab_size, en_vocab_size)


def tokenize(words, word_to_idx):
    return [word_to_idx.get(word, word_to_idx['<UNK>']) for word in words]


processed_data_ch = []
processed_data_en = []

for ch, en in zip(chinese_sentences, english_sentences):
    ch_numerical = [chinese_word_to_idx['<BOS>']] + tokenize(ch, chinese_word_to_idx) + [chinese_word_to_idx['<EOS>']]
    en_numerical = [english_word_to_idx['<BOS>']] + tokenize(en, english_word_to_idx) + [english_word_to_idx['<EOS>']]
    processed_data_ch.append(torch.LongTensor(ch_numerical))
    processed_data_en.append(torch.LongTensor(en_numerical))

print(processed_data_ch)
print(processed_data_en)


In [ ]:
from torch.nn.utils import rnn

processed_data_ch_pad = rnn.pad_sequence(processed_data_ch, batch_first=True,
                                         padding_value=chinese_word_to_idx['<PAD>'])
processed_data_en_pad = rnn.pad_sequence(processed_data_en, batch_first=True,
                                         padding_value=english_word_to_idx['<PAD>'])


In [ ]:
from torch.utils.data import TensorDataset, DataLoader

dataset = TensorDataset(processed_data_ch_pad, processed_data_en_pad)
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

for src, trg in dataloader:
    print(src)
    print(trg)
    break


In [ ]:
import math
from torch import nn


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_seq_len, d_model)
        position = torch.arange(0, max_seq_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:x.size(0), :]
        return x

In [ ]:
class Transformer(nn.Module):
    def __init__(self, d_model, d_ff, n_heads, n_encoder_layers, n_decoder_layers):
        super().__init__()
        self.encoder_embedding = nn.Embedding(vocab_size, d_model)
        self.decoder_embedding = nn.Embedding(vocab_size, d_model)
        self.encoder = Encoder(d_model, d_ff, n_heads, n_encoder_layers)
        self.encoder_positional_encoding = PositionalEncoding(d_model)
        self.decoder_positional_encoding = PositionalEncoding(d_model)
        self.decoder = Decoder(d_model, d_ff, n_heads, n_decoder_layers)
        self.out = nn.Linear(d_model, vocab_size)

    def forward(self, encoder_inputs, decoder_inputs, mask=None):
        encoder_inputs = self.encoder_embedding(encoder_inputs)
        decoder_inputs = self.decoder_embedding(decoder_inputs)
        encoder_inputs = self.encoder_positional_encoding(encoder_inputs.transpose(0, 1)).transpose(0, 1)
        decoder_inputs = self.decoder_positional_encoding(decoder_inputs.transpose(0, 1)).transpose(0, 1)
        encoder_outputs = self.encoder(encoder_inputs)
        decoder_outputs = self.decoder(decoder_inputs, encoder_outputs, mask=mask)
        return self.out(decoder_outputs)



In [ ]:
class Decoder(nn.Module):
    def __init__(self, d_model, d_ff, n_heads, n_decoder_layers):
        super().__init__()
        self.layers = nn.ModuleList([DecoderLayer(d_model, d_ff, n_heads) for _ in range(n_decoder_layers)])

    def forward(self, decoder_inputs, encoder_outputs, mask=None):
        for layer in self.layers:
            decoder_inputs = layer(decoder_inputs, encoder_outputs, mask=mask)
        return decoder_inputs


class DecoderLayer(nn.Module):
    def __init__(self, d_model, d_ff, n_heads):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, d_model, d_model, n_heads)
        self.cross_attn = MultiHeadAttention(d_model, d_model, d_model, n_heads)
        self.ffn = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

    def forward(self, decoder_inputs, encoder_outputs, mask=None):
        attn1 = self.self_attn(q=decoder_inputs, mask=mask)
        x = self.norm1(decoder_inputs + attn1)

        attn2 = self.cross_attn(q=x, k=encoder_outputs, v=encoder_outputs)
        x = self.norm2(x + attn2)

        ffn_out = self.ffn(x)
        x = self.norm3(x + ffn_out)
        return x


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.linear2(torch.relu(self.linear1(x)))

In [ ]:
class Encoder(nn.Module):
    def __init__(self, d_model, d_ff, n_heads, n_encoder_layers):
        super().__init__()
        self.layers = nn.ModuleList([EncoderLayer(d_model, d_ff, n_heads) for _ in range(n_encoder_layers)])

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x


class EncoderLayer(nn.Module):
    def __init__(self, d_model, d_ff, n_heads):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, d_model, d_model, n_heads)
        self.ffn = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        x = self.norm1(x + self.attn(q=x))
        x = self.norm2(x + self.ffn(x))
        return x


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.linear2(torch.relu(self.linear1(x)))


class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim: int, attn_dim: int, output_dim: int, num_heads: int):
        super().__init__()
        self.embed_dim = embed_dim
        self.attn_dim = attn_dim
        self.output_dim = output_dim
        self.num_heads = num_heads
        self.head_dim = attn_dim // num_heads

        self.q_proj = nn.Linear(embed_dim, self.attn_dim, bias=False)
        self.k_proj = nn.Linear(embed_dim, self.attn_dim, bias=False)
        self.v_proj = nn.Linear(embed_dim, self.attn_dim, bias=False)

        self.out_proj = nn.Linear(self.attn_dim, self.output_dim, bias=False)

    def forward(self, q, k=None, v=None, mask=None):
        if k is None: k = q
        if v is None: v = q

        batch_size, seq_len_q, _ = q.shape
        seq_len_k = k.shape[1]

        q = self.q_proj(q).view(batch_size, seq_len_q, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(k).view(batch_size, seq_len_k, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(v).view(batch_size, v.shape[1], self.num_heads, self.head_dim).transpose(1, 2)

        attn_score = torch.matmul(q, k.transpose(-2, -1))
        attn_score = attn_score / torch.sqrt(torch.tensor(self.head_dim, dtype=torch.float32))

        if mask is not None:
            mask = mask.unsqueeze(0).unsqueeze(1)
            attn_score = attn_score.masked_fill(mask == 0, -1e9)

        attn_weight = torch.softmax(attn_score, dim=-1)

        output = torch.matmul(attn_weight, v)
        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len_q, self.attn_dim)
        return self.out_proj(output)

In [ ]:
from torch import optim

model = Transformer(d_model=512, d_ff=2048, n_heads=8, n_encoder_layers=6, n_decoder_layers=6)
optimizer = optim.Adam(model.parameters(), lr=Learning_rate)
criterion = nn.CrossEntropyLoss(ignore_index=english_word_to_idx['<PAD>'])

In [ ]:
epochs = 100

for step in range(epochs):
    for input, target in dataloader:
        decoder_input = target[:, :-1]
        decoder_target = target[:, 1:]

        mask = torch.tril(torch.ones(decoder_input.size(1), decoder_input.size(1)))
        decoder_outputs = model(input, decoder_input, mask)

        loss = criterion(decoder_outputs.view(-1, decoder_outputs.size(-1)), decoder_target.view(-1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f'epoch:{step + 1},loss:{loss.item()}')


In [ ]:
def translate(sentence, model):
    ch_token = torch.LongTensor(tokenize(tokenize_chinese(sentence), chinese_word_to_idx))
    ch_embedding = model.encoder_embedding(ch_token).unsqueeze(0)
    positionalEncoding = PositionalEncoding(512)
    ch_embedding = positionalEncoding(ch_embedding)

    encoder_outputs = model.encoder(ch_embedding)
    decoder_inputs = [english_word_to_idx['<BOS>']]

    for _ in range(10):
        with torch.no_grad():
            decoder_inputs_tensor = torch.LongTensor(decoder_inputs)
            decoder_inputs_tensor = model.decoder_embedding(decoder_inputs_tensor).unsqueeze(0)
            positionalEncoding = PositionalEncoding(512)
            decoder_inputs_tensor = positionalEncoding(decoder_inputs_tensor)
            output = model.decoder(decoder_inputs_tensor, encoder_outputs, mask=None)
            output = model.out(output)
            pred_token = output[:, -1, :].argmax().item()
            decoder_inputs.append(pred_token)
            if pred_token == english_word_to_idx['<EOS>']:
                break
    return ' '.join(english_vocab[idx] for idx in decoder_inputs[1:-1])


test_sentence = '你好，今天天气真好！'

print(translate(test_sentence, model))


### pytorch Transformer

In [ ]:
class MachineTranslation(nn.Module):
    def __init__(self, d_model, d_ff, n_heads, n_encoder_layers, n_decoder_layers):
        super().__init__()
        self.encoder_embedding = nn.Embedding(len(chinese_vocab), d_model)
        self.decoder_embedding = nn.Embedding(len(english_vocab), d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        self.transformer = nn.Transformer(d_model=d_model, dim_feedforward=d_ff, nhead=n_heads,
                                          num_decoder_layers=n_decoder_layers, num_encoder_layers=n_encoder_layers,
                                          batch_first=True)
        self.fc = nn.Linear(d_model, len(english_vocab))

    def forward(self, ch_inputs, en_inputs, mask=None):
        batch_size, en_seq_len = en_inputs.shape
        mask = nn.Transformer.generate_square_subsequent_mask(en_seq_len)

        encoder_inputs = self.pos_encoder(self.encoder_embedding(ch_inputs))
        decoder_inputs = self.pos_encoder(self.decoder_embedding(en_inputs))

        outputs = self.transformer(encoder_inputs, decoder_inputs, tgt_mask=mask)

        return self.fc(outputs)


In [ ]:
import torch.optim as optim

model = MachineTranslation(d_model=512, d_ff=2048, n_heads=8, n_encoder_layers=6, n_decoder_layers=6)
optimizer = optim.Adam(model.parameters(), lr=0.0001)
criterion = nn.CrossEntropyLoss(ignore_index=english_word_to_idx['<PAD>'])

In [ ]:
epochs = 100

for step in range(epochs):
    for input, target in dataloader:
        decoder_input = target[:, :-1]
        decoder_target = target[:, 1:]

        mask = nn.Transformer.generate_square_subsequent_mask(decoder_input.size(1))
        decoder_outputs = model(input, decoder_input, mask)

        loss = criterion(decoder_outputs.view(-1, decoder_outputs.size(-1)), decoder_target.view(-1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f'epoch:{step + 1},loss:{loss.item()}')


In [ ]:
def translate(sentence, model):
    ch_token = torch.LongTensor(tokenize(tokenize_chinese(sentence), chinese_word_to_idx))
    ch_embedding = model.encoder_embedding(ch_token).unsqueeze(0)
    positionalEncoding = PositionalEncoding(512)
    ch_embedding = positionalEncoding(ch_embedding)

    encoder_outputs = model.transformer.encoder(ch_embedding)
    decoder_inputs = [english_word_to_idx['<BOS>']]

    for _ in range(10):
        with torch.no_grad():
            decoder_inputs_tensor = torch.LongTensor(decoder_inputs)
            decoder_inputs_tensor = model.decoder_embedding(decoder_inputs_tensor).unsqueeze(0)
            positionalEncoding = PositionalEncoding(512)
            decoder_inputs_tensor = positionalEncoding(decoder_inputs_tensor)
            mask = nn.Transformer.generate_square_subsequent_mask(decoder_inputs_tensor.size(1))
            output = model.transformer.decoder(decoder_inputs_tensor, encoder_outputs, tgt_mask=mask)
            output = model.fc(output)
            pred_token = output[:, -1, :].argmax().item()
            decoder_inputs.append(pred_token)
            if pred_token == english_word_to_idx['<EOS>']:
                break
    return ' '.join(english_vocab[idx] for idx in decoder_inputs[1:-1])


test_sentence = '你好，今天天气真好！'

print(translate(test_sentence, model))